In [ ]:
####
#  mostrar la importancia de cada variable en una predicción
#  cargar modelo, predecir, waterfall shap
####
import joblib
import json
import sys
import os
import pandas as pd


# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')


####
#  cargar modelo
####
# 1. Define la ruta de la carpeta del experimento que quieres cargar
# (Copia la ruta que se imprimió al guardar)
ruta_experimento = '/content/drive/MyDrive/TFM_Aurea_Team/04_models/lightgbm_FINAL20260206_142533'
# 2. Cargar el modelo (.joblib)
model_path = os.path.join(ruta_experimento, "lightgbm.joblib")
loaded_model = joblib.load(model_path)
# 3. Cargar las métricas y metadatos (.json)
metrics_path = os.path.join(ruta_experimento, "metrics.json")
with open(metrics_path, 'r') as f:
    loaded_metadata = json.load(f)
# --- VERIFICACIÓN ---
print(f"✅ Modelo {loaded_metadata['model_type']} cargado correctamente.")
print(f"📈 AUC en Test: {loaded_metadata['metricas_test']['auc_roc']:.4f}")
print(f"📋 Variables utilizadas: {len(loaded_metadata['variables_usadas'])}")


########################
# Valores de ejemplo de una reserva
LLEGADA = pd.to_datetime('2026-11-06') # Convert to datetime
NOCHES = 17
PAX = 4
ADULTOS = 4
CLIENTE = 'AUREA TRAVEL CLUB_SOCIOS'
FECHA_TOMA = pd.to_datetime('2025-02-22 12:48:29') # Convert to datetime
FIDELIDAD = None
PAIS = 'ESTADOS UNIDOS'
SEGMENTO = 'Fixed rates'
FUENTE_NEGOCIO = 'DIRECT SALES'
NOMBRE_HOTEL = 'Complejo Riviera Maya'
NOMBRE_HABITACION = 'SIG JUNIOR SUITE PS'
VALOR_RESERVA = 4899.40
########################


####
#  preparar reserva
####
# función get_features
module_path = '/content/drive/MyDrive/TFM_Aurea_Team/04_models/'
if module_path not in sys.path:
    sys.path.append(module_path)
from funcion_producion import get_features
new_reservation_df = get_features(
    LLEGADA, NOCHES, PAX, ADULTOS, CLIENTE, FECHA_TOMA, FIDELIDAD, PAIS, SEGMENTO, FUENTE_NEGOCIO, NOMBRE_HOTEL, NOMBRE_HABITACION, VALOR_RESERVA
)

# Identify categorical features from the model's feature names
categorical_features = [
    'FUENTE_NEGOCIO_SEGMENTO_CLIENTE',
    'PAIS_TOP_200',
    'COMPLEJO_RESERVA',
    'TIPO_VIAJERO',
    'HOTEL_HABITACION_TOP'
]
# Convert these columns in new_reservation_df to 'category' dtype
for col in categorical_features:
    if col in new_reservation_df.columns:
        new_reservation_df[col] = new_reservation_df[col].astype('category')



####
#  waterfall shap
####
import shap
explainer = shap.TreeExplainer(loaded_model)
shap_values = explainer(new_reservation_df[loaded_model.feature_name_])
shap.plots.waterfall(shap_values[0])




####
#  predicción
####
# Now, make the prediction
prediction_proba = loaded_model.predict_proba(new_reservation_df[loaded_model.feature_name_])
display(prediction_proba)


¿sí la predicción de este ejemplo es array([[0.40443169, 0.59556831]]), qué significan los valores de la gráfica E[f(X)]=-0.469 y f(x)=0.387?

Let's clarify these values from your SHAP waterfall plot and their relationship to the prediction_proba:

prediction_proba = [[0.40443169, 0.59556831]]:

This is the final output of your model in terms of probabilities for the specific reservation.
0.40443169 is the probability of the reservation not being canceled (Class 0).
0.59556831 is the probability of the reservation being canceled (Class 1). This is the value you would typically interpret as the likelihood of the event occurring.
E[f(X)] = -0.469 (Base Value / Expected Value):

This represents the average raw output of the model across the entire dataset used to train the SHAP explainer. For LightGBM (and many other tree models) on a binary classification task, this 'raw output' is usually in the log-odds (logit) scale before the final sigmoid function transforms it into a probability.
So, -0.469 is the average log-odds of a reservation being canceled in your dataset.
f(x) = 0.387 (Model Output for this Instance):

This represents the raw output of the model for this specific reservation (also in the log-odds scale), after all the individual feature contributions (the bars in the waterfall plot) have been added to the base value E[f(X)].
How they connect:

The f(x) value (log-odds for this instance) is transformed into the final probability using the sigmoid function:

Probability (Class 1) = 1 / (1 + exp(-f(x)))
If we plug in f(x) = 0.387: 1 / (1 + exp(-0.387)) ≈ 1 / (1 + 0.679) ≈ 1 / 1.679 ≈ 0.5956
This 0.5956 matches the 0.59556831 from your prediction_proba perfectly. So, the waterfall plot works on the log-odds scale, showing how features push the prediction from the average log-odds (E[f(X)]) to the specific instance's log-odds (f(x)), which then converts to the final probability you see in prediction_proba.

Explica la diferencia entre los valores SHAP y la importancia de la característica

Both SHAP values and traditional feature importance metrics aim to explain which features are impactful in a model's predictions, but they do so from different perspectives and offer different levels of insight.

Let's break down the difference:

1. SHAP Values (Shapley Additive exPlanations)
Local Explanation: SHAP values explain why a single prediction was made for a specific instance. For example, they can tell you why this particular reservation has a 68.5% chance of cancellation.
Additive Contribution: A SHAP value for a feature quantifies how much that feature's specific value contributed to pushing the model's prediction from the average prediction (the base value E[f(X)]) to the final prediction for that instance (f(x)).
Directional Impact: SHAP values are signed. A positive SHAP value means the feature pushed the prediction higher (e.g., increased cancellation probability), while a negative SHAP value means it pushed it lower (e.g., decreased cancellation probability).
Fairness: SHAP values are based on cooperative game theory and provide a unique, fair, and consistent way to distribute the 'payout' (the prediction difference) among the 'players' (the features).
Contextual: They consider the interaction effects between features, meaning a feature's impact can change depending on the values of other features in that specific instance.
2. Traditional Feature Importance (e.g., feature_importances_ from LightGBM/XGBoost, Permutation Importance)
Global Explanation: Feature importance typically provides an overall ranking or score of how important each feature is to the model as a whole, across the entire dataset. It tells you which features are generally more influential.
Non-Directional: These scores usually do not tell you how a feature influences the prediction (e.g., does a high LEAD_TIME increase or decrease cancellation?). They just tell you it's important.
Calculation Methods Vary:
For tree-based models like LightGBM or XGBoost, it's often based on how many times a feature is used in splits or the total gain (reduction in error) that feature provides across all splits in all trees.
Permutation Importance is a more robust method where a feature's importance is measured by how much the model's performance decreases when the values of that feature are randomly shuffled.
Can be Biased: Some tree-based feature importance methods can be biased towards high-cardinality categorical features or numerical features.
Key Differences Summarized:
Aspect	SHAP Values	Traditional Feature Importance
Scope	Local: Explains single predictions	Global: Explains overall model behavior
Direction	Shows increase/decrease in prediction (signed)	Typically no direction (just magnitude)
Additivity	Contributions sum to the prediction difference	Scores are generally not additive
Consistency	Rigorously defined, theoretically consistent	Can be inconsistent or biased based on method
Interactions	Naturally accounts for feature interactions	Often doesn't explicitly account for interactions
In essence, SHAP values offer a much richer and more granular understanding of a model's decisions, explaining why a specific outcome occurred, while traditional feature importance gives you a general idea of what features are generally important.